In [60]:
import pandas as pd
import graphviz

# DATA LOADING

In [61]:
df_adv = pd.read_csv("../data/processed/data_topic_weights.csv")
df_obs = pd.read_csv("../data/processed/data_obs.csv")
df_obs.drop(df_obs.columns[0], axis=1, inplace=True)
df_obs["condition"] = "OBS"
df_adv["condition"] = "ADV"
df = pd.concat([df_obs, df_adv])
id_to_topic = df.set_index("ID")["assigned_topic"].to_dict()
df["parent_assigned_topic"] = df["parent_ID"].map(id_to_topic)
df.tail()

,ID,understanding_score,pl1_understanding,pl2_understanding,score_corrected,rank,filename,binary_score,pl2_confidence,na_count,...,assigned_topic,parent_topic_0,parent_topic_1,parent_topic_2,parent_topic_3,parent_topic_4,parent_topic_5,parent_topic_6,parent_topic_7,parent_assigned_topic
995,668a935e677dc562a445cd03,0.379820,0.281283,0.478356,0.247881,80,053402_experiment_2024-07-10_19h18.39.977.csv,0.558594,NaN,NaN,...,2.0,0.169108,0.068684,0.330074,0.063610,0.261180,0.008401,0.000000,0.098943,2.0
996,668bb99e5fa8b9ebb3c428ac,0.237887,0.201746,0.274028,-0.082329,91,909045_experiment_2024-07-10_18h48.16.190.csv,0.519531,NaN,NaN,...,7.0,0.484727,0.000000,0.265923,0.000000,0.000000,0.000000,0.000000,0.249351,7.0
997,668bfbcbfd735b6652c60f20,-0.116250,-0.155149,-0.077350,0.732068,61,489637_experiment_2024-07-10_14h22.18.159.csv,0.542969,NaN,NaN,...,2.0,0.133815,0.118867,0.270629,0.089777,0.251170,0.002137,0.037103,0.096502,2.0
998,668d5331aa58b7e75ff160d7,0.091113,0.161720,0.020506,0.714286,63,234906_experiment_2024-07-10_19h00.46.327.csv,0.609375,NaN,NaN,...,1.0,0.047872,0.229925,0.189555,0.279478,0.132181,0.090450,0.030539,0.000000,1.0
999,668d822d705c807ed4664ae1,0.820187,0.642993,0.997381,2.688492,5,564113_experiment_2024-07-10_18h59.42.430.csv,0.730469,NaN,NaN,...,2.0,0.132785,0.038407,0.357090,0.069165,0.281337,0.015952,0.003242,0.102021,2.0


In [62]:
def get_descendants(target_id, df):
    """ Recursively retrieve all descendants of a given target_id """
    descendants = set()
    children = df[df["parent_ID"] == target_id]["ID"]
    
    for child in children:
        descendants.add(child)
        descendants.update(get_descendants(child, df))  # Recursively find their descendants
    
    return descendants


def get_ancestors(target_id, df):
    ancestors = set()
    current = target_id
    while True:
        parent = df.loc[df["ID"] == current, "parent_ID"]
        if parent.empty or pd.isna(parent.values[0]):
            break
        parent = parent.values[0]
        ancestors.add(parent)
        current = parent
    return ancestors

# MS Fig3J

In [ ]:
target_id = "5da83385e8bf0200113c3c6e"  # Replace with the actual ID you want to track

descendant_ids = get_descendants(target_id, df_adv)

df_adv = df[df["condition"] == "ADV"]
# Filter dataset to keep only descendants of the given ID
df_filtered = df_adv[df_adv["ID"].isin(descendant_ids) | (df_adv["ID"] == target_id)]
df_filtered.loc[df_filtered["assigned_topic"] > 5, "assigned_topic"] = -1
df_filtered.loc[df_filtered["parent_assigned_topic"] > 5, "parent_assigned_topic"] = -1

df_filtered.head()

dot = graphviz.Digraph("Fig3J - lineage")
dot.attr('graph', ranksep='40.0', nodesep='2.0')
dot.attr('node', width='10', height='10', fontsize='14', fixedsize='true')

def darken_hex(hex_color, factor=0.8):
    """Darkens a hex color by multiplying RGB values by the factor (0 < factor < 1)."""
    hex_color = hex_color.lstrip('#')
    rgb = [int(hex_color[i:i+2], 16) for i in (0, 2, 4)]
    dark_rgb = [max(0, int(c * factor)) for c in rgb]
    return '#{:02x}{:02x}{:02x}'.format(*dark_rgb)

topic_shapes = {
    -1: "circle",
    0: "triangle",
    1: "square",
    2: "pentagon",
    3: "circle",
    4: "hexagon",
    5: "diamond"
}

topic_colors = {
    -1: "gray",
    # 0: "#F4E285",
    0: "#D4AF37",
    1: "#F4A259",
    2: "#5B8E7D",
    3: "#BC4B51",
    4: "#8CB369",
    5: "#6C5B7B"
}


for row in df_filtered.iterrows():
    ID = row[1]["ID"]
    if ID == target_id:
        continue
    pID = row[1]["parent_ID"]
    topic = row[1]["parent_assigned_topic"]
    color = topic_colors[topic]
    dot.attr("edge", penwidth="80.0", color=color)
    dot.edge(pID, ID)

for row in df_filtered.iterrows():
    ID = row[1]["ID"]
    topic = row[1]["assigned_topic"]
    shape = topic_shapes[topic]
    color = topic_colors[topic]
    dot.node(ID, label="", shape=shape, color=color, style="filled", fillcolor=color, penwidth="20")


pdf_path = dot.render(directory="../results/figures/fig3", format="pdf").replace('\\', '/')

# SM

## LANGUAGE TREATMENT

In [ ]:
df_sm_language = df[df["condition"] == "ADV"]

rows = []

for target_id in df_sm_language["ID"]:
    descendant_ids = get_descendants(target_id, df_sm_language)
    rows.append({
        "ID": target_id,
        "n_descendants": len(descendant_ids),
        "descendants": list(descendant_ids)
    })

df_sm_language = pd.DataFrame(rows)
df_sm_language = df_sm_language.sort_values("n_descendants", ascending=False)
df_sm_language = df_sm_language.query("n_descendants > 50")


candidate_ids = set(df_sm_language["ID"])

pure_rows = []

for _, row in df_sm_language.iterrows():
    ID = row["ID"]
    ancestors = get_ancestors(ID, df)  # IMPORTANT : df complet
    
    # Garder seulement si aucun ancêtre n'est aussi un candidat
    if not (ancestors & candidate_ids):
        pure_rows.append(row)

df_pure = pd.DataFrame(pure_rows).sort_values("n_descendants", ascending=False)


In [66]:
for target_id in df_pure["ID"]:
    descendant_ids = get_descendants(target_id, df_adv)
    if len(descendant_ids) > 0:
        df_adv = df[df["condition"] == "ADV"]
        # Filter dataset to keep only descendants of the given ID
        df_filtered = df_adv[df_adv["ID"].isin(descendant_ids) | (df_adv["ID"] == target_id)]
        df_filtered.loc[df_filtered["assigned_topic"] > 5, "assigned_topic"] = -1
        df_filtered.loc[df_filtered["parent_assigned_topic"] > 5, "parent_assigned_topic"] = -1


        dot = graphviz.Digraph(target_id)
        dot.attr('graph', ranksep='40.0', nodesep='2.0')
        dot.attr('node', width='10', height='10', fontsize='14', fixedsize='true')

        def darken_hex(hex_color, factor=0.8):
            """Darkens a hex color by multiplying RGB values by the factor (0 < factor < 1)."""
            hex_color = hex_color.lstrip('#')
            rgb = [int(hex_color[i:i+2], 16) for i in (0, 2, 4)]
            dark_rgb = [max(0, int(c * factor)) for c in rgb]
            return '#{:02x}{:02x}{:02x}'.format(*dark_rgb)

        topic_shapes = {
            -1: "circle",
            0: "triangle",
            1: "square",
            2: "pentagon",
            3: "circle",
            4: "hexagon",
            5: "diamond"
        }

        topic_colors = {
            -1: "gray",
            # 0: "#F4E285",
            0: "#D4AF37",
            1: "#F4A259",
            2: "#5B8E7D",
            3: "#BC4B51",
            4: "#8CB369",
            5: "#6C5B7B"
        }

        for row in df_filtered.iterrows():
            ID = row[1]["ID"]
            if ID == target_id:
                continue
            pID = row[1]["parent_ID"]
            topic = row[1]["parent_assigned_topic"]
            color = topic_colors[topic]
            dot.attr("edge", penwidth="80.0", color=color)
            dot.edge(pID, ID)

        for row in df_filtered.iterrows():
            ID = row[1]["ID"]
            topic = row[1]["assigned_topic"]
            shape = topic_shapes[topic]
            color = topic_colors[topic]
            dot.node(ID, label="", shape=shape, color=color, style="filled", fillcolor=color, penwidth="20")


        pdf_path = dot.render(directory="../results/figures/SM/lineage/language", format="png").replace('\\', '/')


dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.243434 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.693232 to fit


## OBSERVATION TREATMENT

In [ ]:
df_sm_observation = df[df["condition"] == "OBS"]

rows = []

for target_id in df_sm_observation["ID"]:
    descendant_ids = get_descendants(target_id, df_sm_observation)
    rows.append({
        "ID": target_id,
        "n_descendants": len(descendant_ids),
        "descendants": list(descendant_ids)
    })

df_sm_observation = pd.DataFrame(rows)
df_sm_observation = df_sm_observation.sort_values("n_descendants", ascending=False)
df_sm_observation = df_sm_observation.query("n_descendants > 50")

def get_ancestors(target_id, df):
    ancestors = set()
    current = target_id
    while True:
        parent = df.loc[df["ID"] == current, "parent_ID"]
        if parent.empty or pd.isna(parent.values[0]):
            break
        parent = parent.values[0]
        ancestors.add(parent)
        current = parent
    return ancestors

candidate_ids = set(df_sm_observation["ID"])

pure_rows = []

for _, row in df_sm_observation.iterrows():
    ID = row["ID"]
    ancestors = get_ancestors(ID, df)  # IMPORTANT : df complet
    
    # Garder seulement si aucun ancêtre n'est aussi un candidat
    if not (ancestors & candidate_ids):
        pure_rows.append(row)

df_pure = pd.DataFrame(pure_rows).sort_values("n_descendants", ascending=False)

In [ ]:
for target_id in df_pure["ID"]:
    descendant_ids = get_descendants(target_id, df_obs)
    if len(descendant_ids) > 0:
        # Filter dataset to keep only descendants of the given ID
        df_filtered = df_obs[df_obs["ID"].isin(descendant_ids) | (df_obs["ID"] == target_id)]


        dot = graphviz.Digraph(target_id)
        dot.attr('graph', ranksep='40.0', nodesep='2.0')
        dot.attr('node', width='10', height='10', fontsize='14', fixedsize='true')

        def darken_hex(hex_color, factor=0.8):
            """Darkens a hex color by multiplying RGB values by the factor (0 < factor < 1)."""
            hex_color = hex_color.lstrip('#')
            rgb = [int(hex_color[i:i+2], 16) for i in (0, 2, 4)]
            dark_rgb = [max(0, int(c * factor)) for c in rgb]
            return '#{:02x}{:02x}{:02x}'.format(*dark_rgb)


        for _, row in df_filtered.iterrows():
            ID = row["ID"]
            if ID == target_id:
                continue
            pID = row["parent_ID"]
            dot.attr("edge", penwidth="80.0", color="gray")
            dot.edge(pID, ID)

        for _, row in df_filtered.iterrows():
            ID = row["ID"]
            dot.node(
                ID,
                label="",
                shape="circle",
                color="gray",
                style="filled",
                fillcolor="gray",
                penwidth="20"
            )


        pdf_path = dot.render(directory="../results/figures/SM/lineage/observation", format="png").replace('\\', '/')
